In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_13.pickle

In [4]:
%%RecordEvent
%%time
### cell 13 ###
def grab_subset_of_data_25(df, question_of_interest):
    # Select only the columns whose names contain the question (GPU-side via boolean mask)
    mask = df.columns.str.contains(question_of_interest)
    sub_df = df.loc[:, mask]

    # Remove everything up to the last dash and both unwanted substrings in one GPU call
    pattern = r".*-\s*|\s*\(direct from AWS, Azure, GCP, or similar\)|\s*\(resulting in a university degree\)"
    sub_df.columns = sub_df.columns.str.replace(pattern, "", regex=True)

    # Drop rows where all values in these columns are null using GPU operations
    non_null_rows = sub_df.notna().any(axis=1)
    return sub_df.loc[non_null_rows]

question_of_interest_cell25 = 'On which platforms have you begun or completed data science courses?'
online_learning_platforms_df_2022 = grab_subset_of_data_25(
    responses_df_2022_cell10,
    question_of_interest_cell25
)

online_learning_platforms_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21354 entries, 0 to 23996
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Coursera                      9699 non-null   object 
 1   edX                           2474 non-null   object 
 2   Kaggle Learn Courses          6628 non-null   object 
 3   DataCamp                      3718 non-null   object 
 4   Fast.ai                       944 non-null    object 
 5   Udacity                       2199 non-null   object 
 6   Udemy                         6116 non-null   object 
 7   LinkedIn Learning             2766 non-null   object 
 8   Cloud—certification programs  1821 non-null   object 
 9   University Courses            6780 non-null   object 
 10  None                          0 non-null      float64
 11  Other                         5669 non-null   object 
dtypes: float64(1), object(11)
memory usage: 2.1+ MB
CPU times: user 2

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_13_try_2.pickle